# Aggiungi il tool SQL
Copia le celle nel notebook Investigatore dopo genera. Carica lo ZIP del giorno 4 con la cella di caricamento: contiene accessi.sqlite. Questo tool legge soltanto il database. Modificate il prompt o gestite gli errori; mostrate la query e le righe usate.


In [ ]:
"""Tool di sola lettura sul database di esercitazione."""
import sqlite3
from pathlib import Path

SCHEMA='''operatori(id, nome)
depositi(id, nome, settore)
accessi(id, operatore_id, deposito_id, pratica, codice, stato, valido_fino)
accessi.operatore_id collega operatori.id; accessi.deposito_id collega depositi.id.
Il codice è testo: conserva eventuali zeri iniziali.
Data della missione: 2031-10-12 06:00. stato deve essere attivo, valido_fino almeno pari a questa data.
I dati richiesti per la ricerca arrivano dai dossier. Se mancano, chiedili all’utente.'''

def interroga_database(query, database='accessi.sqlite'):
    if not isinstance(query,str) or not query.strip():return {'errore':'Scrivi una query SQL.'}
    path=Path(database).resolve()
    if not path.is_file():return {'errore':'Carica il database della tua squadra.'}
    con=sqlite3.connect(path.as_uri()+'?mode=ro',uri=True)
    con.execute('PRAGMA query_only=ON')
    tables={'operatori','depositi','accessi'}
    def authorize(action,arg1,arg2,db,trigger):
        if action==sqlite3.SQLITE_SELECT:return sqlite3.SQLITE_OK
        if action==sqlite3.SQLITE_READ:return sqlite3.SQLITE_OK if arg1 in tables else sqlite3.SQLITE_DENY
        if action==sqlite3.SQLITE_FUNCTION:return sqlite3.SQLITE_OK if (arg2 or '').lower() in {'count','min','max','lower','upper','length','coalesce','sum','avg'} else sqlite3.SQLITE_DENY
        return sqlite3.SQLITE_DENY
    con.set_authorizer(authorize)
    ticks=0
    def limit():
        nonlocal ticks
        ticks+=1
        return 1 if ticks>1000 else 0
    con.set_progress_handler(limit,1000)
    try:
        cursor=con.execute(query)
        rows=cursor.fetchmany(51)
        return {'colonne':[x[0] for x in cursor.description],'righe':[list(r) for r in rows[:50]],'altre_righe':len(rows)>50}
    except sqlite3.Error as e:return {'errore':str(e)}
    finally:con.close()


In [ ]:
# Prova tecnica: questa query non risolve la missione.
print(interroga_database('SELECT nome FROM operatori LIMIT 3'))


In [ ]:
import json
def agente_sql(domanda):
    richiesta=genera([
        {'role':'system','content':'Rispondi solo con un oggetto JSON nella forma {"sql":"SELECT ..."}. Scrivi una sola query di lettura. Schema: '+SCHEMA},
        {'role':'user','content':domanda}
    ],max_tokens=280)
    try:
        testo=richiesta.strip()
        if testo.startswith('```'):testo='\n'.join(testo.splitlines()[1:-1])
        proposta=json.loads(testo)
        if set(proposta)!={'sql'} or not isinstance(proposta['sql'],str):raise ValueError('Formato inatteso')
    except (ValueError,TypeError):
        return 'Il modello non ha prodotto SQL valido. Riformula la domanda. Output: '+richiesta
    esito=interroga_database(proposta['sql'])
    if 'errore' in esito:return 'Query proposta: '+proposta['sql']+'\nErrore: '+esito['errore']
    sintesi=genera([{'role':'system','content':'Spiega solo il risultato del database. Non inventare righe o codici. Conserva gli zeri iniziali.'},{'role':'user','content':json.dumps(esito,ensure_ascii=False)}])
    return 'QUERY\n'+proposta['sql']+'\nRISULTATO\n'+json.dumps(esito,ensure_ascii=False)+'\nSPIEGAZIONE\n'+sintesi


In [ ]:
import gradio as gr
strumento_sql=gr.Interface(fn=agente_sql,inputs=gr.Textbox(label='Domanda con gli indizi trovati'),outputs=gr.Textbox(label='Query e risultato',lines=16),title='LAZZARO — accessi al deposito')
strumento_sql.launch()


## Dimostra il collegamento
Fai una domanda in linguaggio naturale con persona, deposito, settore e pratica trovati. Il modello propone SQL; il codice chiama il tool; il risultato torna al modello. La query a mano serve per correggere errori, ma nella demo deve partire dalla domanda all’assistente. Non basta leggere il database da soli.
